In [1]:
import ast
import glob
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
all_data = []

for genus_name in keep_genus:
    anno_dir = rf'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/annotations'
    prediction_re = pd.read_csv(f"{anno_dir}/NMS_replicon_type_prediction_results.csv")
    prediction_re['genus'] = genus_name
    all_data.append(prediction_re)

all_data = pd.concat(all_data, ignore_index=True)

In [3]:
name_dict = {'contig_name': 'contig',
             'average plasmid fraction-pident_90': 'plasmidness-pident_90',
             'plasflow_pdc': 'plasflow_prediction',
             'plasmer_pdc': 'plasmer_prediction',
             'rfplasmid_pdc': 'rfplasmid_prediction',
             'deeplasmid_pdc': 'deeplasmid_prediction',
            }
all_data.rename(columns=name_dict, inplace=True)
all_data = all_data[['contig', 'genus', 'size', 'plasmidness-pident_90', 'category-pident_90',
       'initial_type', 'plasflow_prediction', 'plasmer_prediction', 'rfplasmid_prediction',
       'deeplasmid_prediction']]
all_data = all_data[all_data['category-pident_90'].isin(['typical plasmid', 'intermediate replicon'])]

target_dir = '/active-data/analysis_results/chr_pla/genus/suptables'
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)
all_data.to_csv('replicon_category_prediction_data.tsv', sep='\t', index=False)